# || NEMO Additional 2D and 3D Modules ||
© Konstantinos Andreadis 2024 (PhD in the Roux Lab & Salbreux Lab at UNIGE, Switzerland)

please cite: K.Andreadis _et al._ "NEMO: Mesh-Based Tangential Nematic Field, Defect, and Morphology Analysis in Volumetric Microscopy Data" (2026, _in preparation_)

In [ ]:
%load_ext autoreload
%autoreload 2
# Import custom module_scripts

from module_scripts import analysis, datahandler, visuals
from extra import extra_nematic

# Import python essentials
import os
import numpy as np
import matplotlib.pyplot as plt

# --Import Image--

In [ ]:
# ==== Choose Image ====
# [!] WINDOWS: Sometimes the r before the file path string is needed, no idea why.
img_path = '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/nemo_debug.tif'
print(f"Selected image path: {img_path}")

# ==== Choose Time Step and Channel ====
t_select = 0
c_select = 0

# ==== [Optional] Reduce Resolution ====
z_reduce_factor = 1  # 1 means no reduction
xy_reduce_factor = 1  # 1 means no reduction

# ==== [Optional] Normalise Intensities to [0, 1] ====
normalise_intensities = False  # can be set to False

# ==== [Optional] Overwrite Scaling with FIJI Values ====
custom_scaling = None  # please use (z, y, x)

# ==== Load Image ====
img_load = analysis.load_img_virtual(path=img_path, norm_vals=normalise_intensities, t_sel_idx=t_select,
                                     c_sel_idx=c_select, custom_scaling=custom_scaling, reduce_xy=xy_reduce_factor,
                                     reduce_z=z_reduce_factor)
if img_load is not None:
    img_raw, img_dim, img_scale, img_unit = img_load

    # ==== Create Folder Structure ====
    resdata_dir, resfig_dir = datahandler.create_resdirs(img_path, ct_label=f"t={t_select}_c={c_select}")

    # ==== Plot Image Slices and Max Projections ====
    visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, max_proj=True, cmap='Greens',
                     savefig=os.path.join(resfig_dir, "sliced_maxproj_raw.pdf"))
    # z_i, y_i, x_i = int(200 / img_scale[0]), int(570 / img_scale[1]), int(455 / img_scale[2])
    z_i, y_i, x_i = int(img_dim[0] // 2), int(img_dim[1] // 2), int(img_dim[2] // 2)
    visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, x_i=x_i, y_i=y_i, z_i=z_i,
                     savefig=os.path.join(resfig_dir, "sliced_raw.pdf"), cmap="Greens_r")
else:
    # ==== Default scenario ====
    img_raw, img_dim, img_scale, img_unit = np.zeros((1, 1, 1)), (1, 1, 1), (1, 1, 1), "?"
    resdata_dir, resfig_dir = None, None

# -- Nematic Extraction & Analysis --

## |1| 2D Slice

### |1.1| Directors + S order

In [ ]:
# ==== Choose Slices for Analysis ====
sel_slices = np.arange(img_raw.shape[0])
sel_slices = [img_dim[0] // 2]
mask_thresh_val = 0.2 * np.min(analysis.yen_thresh(img_raw[sel_slices]))
img_selected = img_raw[img_dim[0] // 2].copy()

# ==== Saving Parameters  ====
figsize = (4, 4)
resfig_dir_2dsliced = os.path.join(resfig_dir, "2d-sliced_analysis")
resdata_dir_2dsliced = os.path.join(resdata_dir, "2d-sliced_analysis")
datahandler.create_dir(resfig_dir_2dsliced)
datahandler.create_dir(resdata_dir_2dsliced)

# ==== Analysis Parameters ====
boxsize = 2 * 3
patch_avg = ["nearest", 5 ** 2]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

results_2d_slice = []

# ==== Enable Plotting ====
plot_figs = True

for i in sel_slices:
    print(f">> Analysing Z slice {i}")
    img_selected = img_raw[i].copy()
    # ==== 2D Orientation Analysis ====
    orient2d_results = analysis.orient2d(img=img_selected, boxsize=boxsize, thresh_val=mask_thresh_val,
                                         num_neigh_nem=patch_size,
                                         debug=False)
    theta_all_deg, theta_masked_rad, S_2d, n_avg_2d, X, Y, directors_2d = orient2d_results

    directors_2d[:, 0] *= img_scale[1]
    directors_2d[:, 1] *= img_scale[2]
    directors_2d[:, :2] = directors_2d[:, [1, 0]]
    directors_2d_avg_local = np.column_stack((directors_2d[:, :2], n_avg_2d))
    results_2d_slice.append([i, np.nanmean(theta_all_deg), np.nanmean(S_2d)])
    interval = 500
    directors_2d, directors_2d_avg_local, S_2d = directors_2d[::interval], directors_2d_avg_local[::interval], S_2d[
        ::interval]
    print(f"Found {len(directors_2d)} directors !")

    # ==== Save Directors and Order ====
    datahandler.save_array(directors_2d, f"directors_2d_z-{i}", header="x,y,vx,vy", folderpath=resdata_dir_2dsliced)
    datahandler.save_array(S_2d, f"S-order_2d_{patch_label}_z-{i}", header="S", folderpath=resdata_dir_2dsliced)
    datahandler.save_array(directors_2d_avg_local, f"directors-avg_2d_{patch_label}_z-{i}",
                           header="x,y,vx,vy", folderpath=resdata_dir_2dsliced)
    if plot_figs:
        # ==== Plot Results ====
        savefig_i = os.path.join(resfig_dir_2dsliced, f"z-{i}.pdf")
        savefig_theta = os.path.join(resfig_dir_2dsliced, f"z-{i}_theta.pdf")
        savefig_i_dirs = os.path.join(resfig_dir_2dsliced, f"z-{i}_dirs")
        savefig_dirs_s = os.path.join(resfig_dir_2dsliced, f"z-{i}_dirs_s")
        savefig_theta_hist_polar = os.path.join(resfig_dir_2dsliced, f"z_{i}_theta_hist.pdf")

        visuals.plot_matrix(img_selected, scale=img_scale, cmap='Greys_r', colorbar=True,
                            title='Intensity Profile', unit=img_unit,
                            figsize=figsize, savefig=savefig_i)
        visuals.plot_matrix(theta_all_deg, scale=img_scale, cmap='twilight', colorbar=True,
                            unit=img_unit,
                            title=f'Theta Angles for block {boxsize}x{boxsize}', figsize=figsize,
                            savefig=savefig_theta,
                            cmap_limits=[-90, 90])
        visuals.plot_polar_hist(angles_deg=theta_all_deg, savefig=savefig_theta_hist_polar, figsize=figsize)
        visuals.plot_hist(array=S_2d, title="order parameter $S$", figsize=figsize, savefig=savefig_dirs_s)

        # visuals.plot_matrix_vectors(Y, X, theta_masked_rad.T, img_selected, title="Intensity Profile + Directors",
        #                             figsize=(figsize[0] * 2, figsize[1]),
        #                             veclength=10, savefig=savefig_i_dirs, vec_colors=S_2d,
        #                             scale=img_scale,
        #                             cbar_matrix_label="Intensity Signal (a.u.)",
        #                             cbar_vector_label="order parameter $S$")

results_2d_slice = np.array(results_2d_slice)

### -- Load Segmentation Results --

In [ ]:
# patch_avg = ["radius", 5]
patch_avg = ["nearest", 6]
patch_type = patch_avg[0]
patch_size = patch_avg[1]
if not patch_type in ["radius", "nearest"]:
    print(f"[!] Unknown patch type: {patch_type}")

crop_max_ar = 8.0  # Max for colorbar !
local_S_weighted_max = 3.0  # Max for colorbar !
xy_pixel_scale = np.mean(img_scale[1:])
file_ending = "png"

z_range_2d_analysis = np.arange(img_dim[0])

hide_all_figs = True
dpi_all_figs = 100
figsize = (22, 10)
resfig_dir_2dsliced = os.path.join(resfig_dir, "2d-sliced_analysis")
resdata_dir_2dsliced = os.path.join(resdata_dir, "2d-sliced_analysis")
datahandler.create_dir(resfig_dir_2dsliced)
datahandler.create_dir(resdata_dir_2dsliced)
img_name = os.path.basename(img_path).split(".tif")[0]

In [ ]:
directors_2d_all = []
aspect_ratio_all = []
valid_2dseg_idxs = []
masks = mask_outlines = None
only_load_ellip = False

for z_sel in z_range_2d_analysis:
    print(f">> Z-slice selected: {z_sel} / {z_range_2d_analysis.max()}")

    # ==== Load Slice Segmentation Results ====
    # seg_ellips_results = datahandler.load_array(name=f"{z_sel + 1:04d}_seg-ellipses", folderpath=resdata_dir_2dsliced,
    #                                             return_df=True)
    # seg_ellips_results = datahandler.load_array(name=f"{img_name}_z{z_sel + 1}_cp_masks_Ellipsoids",
    #                                             folderpath=resdata_dir_2dsliced,
    #                                             return_df=True)

    seg_ellips_results = datahandler.load_array(name=f"{img_name}_z{z_sel + 1}_cp_masks_ellipses",
                                                folderpath=os.path.join(os.path.dirname(img_path),
                                                                        "z_slice_segmentation", img_name),
                                                return_df=True)
    # ==== Process (or skip empty) Slice Segmentations ====
    if seg_ellips_results is not None and len(seg_ellips_results) > 1:
        seg_ellips_results["Ellipse.Orientation"] *= -1
        seg_ellips_results["Ellipse.Radius1"] *= xy_pixel_scale
        seg_ellips_results["Ellipse.Radius2"] *= xy_pixel_scale
        seg_ellips_results["aspect_ratio"] = seg_ellips_results["Ellipse.Radius1"] / seg_ellips_results[
            "Ellipse.Radius2"]
        valid_2dseg_idxs.append(z_sel)

        # ==== [!!] ONLY DEBUG [!] ====
        # seg_ellips_results["Ellipse.Orientation"] = np.ones(len(seg_ellips_results["Ellipse.Orientation"])) * 0  # !!!!!!
        # seg_ellips_results["Ellipse.Orientation"] = np.random.uniform(-180, 180, size=len(seg_ellips_results["Ellipse.Orientation"]))
        # seg_ellips_results["aspect_ratio"] = np.ones(len(seg_ellips_results["aspect_ratio"])) * 2
        seg_ellips_results = seg_ellips_results[seg_ellips_results["aspect_ratio"] < crop_max_ar]
        if len(seg_ellips_results) <= 1:
            print(f"No segmentation found for {z_sel}, continuing to next z slice...")
            continue
        # ==== [!!] ONLY DEBUG [!] ====

        if not only_load_ellip:
            # ==== Load Segmentation Masks ====
            # masks = datahandler.load_png(os.path.join(resdata_dir_2dsliced, f"{z_sel + 1:04d}_seg-masks.pdf"))
            # masks = datahandler.load_png(os.path.join(resdata_dir_2dsliced, f"{img_name}_z{z_sel + 1}_cp_masks.pdf"))
            masks = datahandler.load_png(os.path.join(os.path.dirname(img_path), "z_slice_segmentation", img_name,
                                                      f"{img_name}_z{z_sel + 1}_cp_masks.pdf"))

            if masks is not None:
                mask_outlines = analysis.contour_masks(masks)
            else:
                print(f"No masks found for {z_sel}, continuing to next z slice...")
                continue
    else:
        print(f"No segmentation found for {z_sel}, continuing to next z slice...")
        continue

    # ==== Saving Path Initialisation ====
    resfig_dir_2dsliced = os.path.join(resfig_dir, "2d-sliced_analysis")
    resdata_dir_2dsliced = os.path.join(resdata_dir, "2d-sliced_analysis")
    datahandler.create_dir(resfig_dir_2dsliced)
    datahandler.create_dir(resdata_dir_2dsliced)

    # ==== Convert Orientations to Director Field ====
    seg_directors_2d = np.column_stack((seg_ellips_results["Ellipse.Center.X"],
                                        seg_ellips_results["Ellipse.Center.Y"],
                                        np.cos(np.radians(seg_ellips_results["Ellipse.Orientation"])),
                                        np.sin(np.radians(seg_ellips_results["Ellipse.Orientation"]))))
    print("Max aspect ratio =", np.max(seg_ellips_results["aspect_ratio"]))
    seg_directors_2d[:, :2] *= img_scale[1:]
    directors_2d = seg_directors_2d.copy()
    directors_2d_all.append(directors_2d)
    aspect_ratio_all.append(np.array(seg_ellips_results["aspect_ratio"]))

    if not only_load_ellip:
        # ==== Local (Weighted) Nematic Order ====
        seg_ellips_results["shape_scalar"] = seg_ellips_results["aspect_ratio"] - 1

        if patch_type == "radius":
            local_idxs = analysis.coord_search_radius(directors_2d[:, :2], r=patch_size)
            patch_label = f"r-{patch_size}{img_unit}"
            title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
            title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
        elif patch_type == "nearest":
            if len(directors_2d) <= patch_size:
                print(f"Only {len(directors_2d)} directors, using k=2 ...")
                patch_size = 2
            local_idxs = analysis.coord_search_neighbours(directors_2d[:, :2], k=patch_size, n_process=8)
            patch_label = f"k-{patch_size - 1}"
            title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
            title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
        else:
            local_idxs = patch_label = title_hist = title_render = None
            print(f"[!] Unknown patch type: {patch_type}")

        S_2d_seg_local, n_2d_seg_local = analysis.avg_2d_nem_tens(directors_2d, weights=None,
                                                                  neigh_idxs=local_idxs)
        S_2d_seg_local_weighted, n_2d_seg_local = analysis.avg_2d_nem_tens(directors_2d, local_idxs,
                                                                           weights=np.array(
                                                                               seg_ellips_results["shape_scalar"]))
        print(f"Max weighted order S: {S_2d_seg_local_weighted.max()}")
        patch_size = patch_avg[1]

        # ==== Plot Results ====
        z_sel_label = f"| Z = {z_sel} / {img_dim[0] - 1}"
        visuals.plot_matrix(matrix=img_raw[z_sel],
                            scale=img_scale, unit=img_unit, figsize=figsize,
                            cmap_label="Intensity Signal (a.u.)",
                            title=f"Raw Image {z_sel_label}", cmap="gray", colorbar=True,
                            savefig=os.path.join(resfig_dir_2dsliced, f"z-{z_sel}_raw.{file_ending}"),
                            hidefig=hide_all_figs, dpi=dpi_all_figs)
        visuals.plot_img_2d_masks(matrix=img_raw[z_sel], unit=img_unit, scale=img_scale,
                                  segmentation_mask=(masks, mask_outlines),
                                  figsize=figsize, cmap="tab20", alpha=0.8, title=f"Segmentation Masks {z_sel_label}",
                                  savefig=os.path.join(resfig_dir_2dsliced, f"z-{z_sel}_seg_raw_masks.{file_ending}"),
                                  hidefig=hide_all_figs, dpi=dpi_all_figs)
        visuals.plot_polar_hist(seg_ellips_results["Ellipse.Orientation"],
                                savefig=os.path.join(resfig_dir_2dsliced,
                                                     f"z-{z_sel}_seg_thetas_polar-hist.{file_ending}"),
                                hidefig=hide_all_figs, dpi=dpi_all_figs, title=f"Orientation Angles {z_sel_label}")

        long_axis_mean = np.mean(seg_ellips_results["Ellipse.Radius1"])
        visuals.plot_hist(seg_ellips_results["Ellipse.Radius1"],
                          title=fr"Long Axis with mean$\approx${long_axis_mean:.2f} {img_unit} {z_sel_label}",
                          savefig=os.path.join(resfig_dir_2dsliced, f"z-{z_sel}_seg_long-axis_hist.{file_ending}"),
                          hidefig=hide_all_figs, dpi=dpi_all_figs)
        short_axis_mean = np.mean(seg_ellips_results["Ellipse.Radius2"])
        visuals.plot_hist(seg_ellips_results["Ellipse.Radius2"],
                          title=fr"Short Axis with mean$\approx${short_axis_mean:.2f} {img_unit} {z_sel_label}",
                          savefig=os.path.join(resfig_dir_2dsliced, f"z-{z_sel}_seg_short-axis_hist.{file_ending}"),
                          hidefig=hide_all_figs, dpi=dpi_all_figs)
        if crop_max_ar is None:
            ar_plot_range = None
        else:
            ar_plot_range = [0, crop_max_ar]

        visuals.plot_hist(seg_ellips_results["aspect_ratio"], title=f"Aspect Ratio {z_sel_label}",
                          savefig=os.path.join(resfig_dir_2dsliced, f"z-{z_sel}_seg_ar_hist.{file_ending}"),
                          hidefig=hide_all_figs, xlim=ar_plot_range, dpi=dpi_all_figs)
        visuals.plot_matrix_vectors(x=seg_ellips_results["Ellipse.Center.X"], y=seg_ellips_results["Ellipse.Center.Y"],
                                    angle_field=np.radians(seg_ellips_results["Ellipse.Orientation"]),
                                    matrix=img_raw[z_sel], veclength=15 * np.array(seg_ellips_results["aspect_ratio"]),
                                    scale=img_scale, unit=img_unit, figsize=figsize,
                                    vec_colors=seg_ellips_results["aspect_ratio"],
                                    cbar_matrix_label="Intensity Signal (a.u.)",
                                    cbar_vector_label="Aspect Ratio",
                                    title=f"Segmented Ellipse Long Axes {z_sel_label}",
                                    savefig=os.path.join(resfig_dir_2dsliced, f"z-{z_sel}_seg_raw_ar.{file_ending}"),
                                    hidefig=hide_all_figs, vec_cmap_limits=ar_plot_range, dpi=dpi_all_figs)
        # NORMALISED
        visuals.plot_matrix_vectors(x=seg_ellips_results["Ellipse.Center.X"], y=seg_ellips_results["Ellipse.Center.Y"],
                                    angle_field=np.radians(seg_ellips_results["Ellipse.Orientation"]),
                                    matrix=img_raw[z_sel], veclength=15 * np.array(seg_ellips_results["aspect_ratio"]),
                                    scale=img_scale, unit=img_unit, figsize=figsize, vec_colors=S_2d_seg_local,
                                    vec_cmap_limits=[0, 1], cbar_matrix_label="Intensity Signal (a.u.)",
                                    cbar_vector_label="Local Nematic Order $S$",
                                    title=f"Local Nematic Order {patch_label} {z_sel_label}",
                                    savefig=os.path.join(resfig_dir_2dsliced,
                                                         f"z-{z_sel}_seg_raw_local_S_{patch_label}.{file_ending}"),
                                    hidefig=hide_all_figs, dpi=dpi_all_figs)
        if len(seg_ellips_results) > 4:
            visuals.plot_slice_heatmap(coords=directors_2d[:, :2], values=S_2d_seg_local, pt_size=5,
                                       cmap_label="Local Nematic Order",
                                       title=f"Local Nematic Order {patch_label} {z_sel_label}",
                                       img_dim=img_dim, img_scale=img_scale, manual_vminvmax=[0, 1],
                                       savefig=os.path.join(resfig_dir_2dsliced,
                                                            f"z-{z_sel}_seg_local_S_heatmap_{patch_label}.{file_ending}"),
                                       hidefig=hide_all_figs, dpi=dpi_all_figs)
        visuals.plot_hist(S_2d_seg_local, title=f"Local Nematic Order $S$ {patch_label} {z_sel_label}",
                          savefig=os.path.join(resfig_dir_2dsliced,
                                               f"z-{z_sel}_seg_local_S_hist_{patch_label}.{file_ending}"),
                          hidefig=hide_all_figs, xlim=[0, 1], dpi=dpi_all_figs)
        # WEIGHTED
        visuals.plot_matrix_vectors(x=seg_ellips_results["Ellipse.Center.X"], y=seg_ellips_results["Ellipse.Center.Y"],
                                    angle_field=np.radians(seg_ellips_results["Ellipse.Orientation"]),
                                    matrix=img_raw[z_sel], veclength=15 * np.array(seg_ellips_results["aspect_ratio"]),
                                    scale=img_scale, unit=img_unit, figsize=figsize, vec_colors=S_2d_seg_local_weighted,
                                    cbar_matrix_label="Intensity Signal (a.u.)",
                                    vec_cmap_limits=[0, local_S_weighted_max],
                                    cbar_vector_label="Local Weighted Nematic Order $S$",
                                    title=f"Local Weighted Nematic Order {patch_label} {z_sel_label}",
                                    savefig=os.path.join(resfig_dir_2dsliced,
                                                         f"z-{z_sel}_seg_raw_local_S_weighted_{patch_label}.{file_ending}"),
                                    hidefig=hide_all_figs, dpi=dpi_all_figs)
        if len(seg_ellips_results) > 4:
            visuals.plot_slice_heatmap(coords=directors_2d[:, :2], values=S_2d_seg_local_weighted,
                                       cmap_label="Local Weighted Nematic Order", pt_size=5,
                                       title=f"Local Weighted Nematic Order {patch_label} {z_sel_label}",
                                       img_dim=img_dim, img_scale=img_scale, manual_vminvmax=[0, local_S_weighted_max],
                                       savefig=os.path.join(resfig_dir_2dsliced,
                                                            f"z-{z_sel}_seg_local_S_weighted_heatmap_{patch_label}.{file_ending}"),
                                       hidefig=hide_all_figs, dpi=dpi_all_figs)
        visuals.plot_hist(S_2d_seg_local_weighted,
                          title=f"Local Weighted Nematic Order $S$ {patch_label} {z_sel_label}",
                          savefig=os.path.join(resfig_dir_2dsliced,
                                               f"z-{z_sel}_seg_local_S_weighted_hist_{patch_label}.{file_ending}"),
                          hidefig=hide_all_figs, xlim=[0, local_S_weighted_max], dpi=dpi_all_figs)
valid_2dseg_idxs = np.array(valid_2dseg_idxs, dtype=int)

In [ ]:
# ==== Save Result Figures as .gif ====
# datahandler.save_gif_multiple(folderpath=os.path.join(resfig_dir, "2d-sliced_analysis"))
datahandler.save_video_multiple(folderpath=os.path.join(resfig_dir, "2d-sliced_analysis"), ext="mp4")

### (experimental) 3D Reconstruction

In [ ]:
# valid_2dseg_idxs = []
# dirs_test = []
# ar_test = []
# masks_all = []
# for z_sel in z_range_2d_analysis:
#     print(f">> Z-slice selected: {z_sel} / {z_range_2d_analysis.max()}")
#
#     # ==== Load Slice Segmentation Results ====
#     seg_ellips_results = datahandler.load_array(name=f"{img_name}_z{z_sel + 1}_cp_masks_Ellipsoids",
#                                                 folderpath=resdata_dir_2dsliced, return_df=True)
#
#     if seg_ellips_results is not None and len(seg_ellips_results) > 1:
#         seg_ellips_results["Ellipse.Orientation"] *= -1
#         seg_ellips_results["Ellipse.Radius1"] *= xy_pixel_scale
#         seg_ellips_results["Ellipse.Radius2"] *= xy_pixel_scale
#         seg_ellips_results["aspect_ratio"] = seg_ellips_results["Ellipse.Radius1"] / seg_ellips_results[
#             "Ellipse.Radius2"]
#         seg_ellips_results = seg_ellips_results[seg_ellips_results["aspect_ratio"] < 5.0]
#         # ==== Load Segmentation Masks ====
#         masks = datahandler.load_png(os.path.join(resdata_dir_2dsliced, f"{img_name}_z{z_sel + 1}_cp_masks.pdf"))
#         if masks is not None:
#             masks_all.append(masks)
#             valid_2dseg_idxs.append(z_sel)
#         else:
#             print(f"No masks found for {z_sel}, continuing to next z slice...")
#             continue
#     else:
#         print(f"No segmentation found for {z_sel}, continuing to next z slice...")
#         continue
#
#     # ==== Saving Path Initialisation ====
#     resfig_dir_2dsliced = os.path.join(resfig_dir, "2d-sliced_analysis")
#     resdata_dir_2dsliced = os.path.join(resdata_dir, "2d-sliced_analysis")
#     datahandler.create_dir(resfig_dir_2dsliced)
#     datahandler.create_dir(resdata_dir_2dsliced)
#
#     # ==== Convert Orientations to Director Field ====
#     seg_directors_2d = np.column_stack((seg_ellips_results["Ellipse.Center.X"],
#                                         seg_ellips_results["Ellipse.Center.Y"],
#                                         np.cos(np.radians(seg_ellips_results["Ellipse.Orientation"])),
#                                         np.sin(np.radians(seg_ellips_results["Ellipse.Orientation"]))))
#     print("Max aspect ratio =", np.max(seg_ellips_results["aspect_ratio"]))
#     seg_directors_2d[:, :2] *= img_scale[1:]
#     directors_2d = seg_directors_2d.copy()
#     ar_test.append(np.array(seg_ellips_results["aspect_ratio"]))
#     dirs_test.append(directors_2d)
#
# dirs_test = np.concatenate(dirs_test)
# zpos = np.concatenate([np.ones_like(ar_test[i]) * idx * img_scale[0] for i, idx in enumerate(valid_2dseg_idxs)])
# ar_test = np.concatenate(ar_test)
# masks_all = np.stack(masks_all)
# valid_2dseg_idxs = np.array(valid_2dseg_idxs)
# full_masks = np.zeros(shape=img_dim, dtype=np.int32)
# full_masks[valid_2dseg_idxs] = masks_all

In [ ]:
# # ==== Render 2D Segmentations in 3D ====
# directors_2dplus = np.column_stack(
#     (zpos, dirs_test[:, 1], dirs_test[:, 0],
#      np.zeros(len(dirs_test)), -dirs_test[:, 3] * ar_test, dirs_test[:, 2] * ar_test))
# visuals.view_3d_vector_field(vec_pos=directors_2dplus[:, :3], vec_dir=directors_2dplus[:, 3:],
#                              vec_colors=visuals.color_scalar(ar_test, manual_vminmax=[0, crop_max_ar]), length=3,
#                              img=img_raw, scale=img_scale)


In [ ]:
# plt.figure()
# plt.hist2d(x=directors_2dplus[:, 0], y=ar_test, density=True)
# plt.colorbar()
# plt.xlabel(f"Z position ({img_unit})")
# plt.ylabel("Aspect ratio")
# plt.show()
#
# plt.figure()
# plt.hist2d(x=directors_2dplus[:, 1], y=ar_test, density=True)
# plt.colorbar()
# plt.xlabel(f"Y position ({img_unit})")
# plt.ylabel("Aspect ratio")
# plt.show()
# z_i, y_i, x_i = int(img_dim[0] // 2), int(img_dim[1] // 2), int(img_dim[2] // 2)
# visuals.plot_img(img=img_raw, scale=img_scale, unit=img_unit, x_i=x_i, y_i=y_i, z_i=z_i,
#                  cmap="Greens_r")

In [ ]:
# visuals.view_colored_labels_3d(masks_all, scale=img_scale, img=img_raw)

In [ ]:
# stitched_3d_plotted = stitched_3d.copy()
# stitched_3d_plotted[stitched_3d_plotted == 28] = 0
# visuals.view_colored_labels_3d(stitched_3d_plotted, scale=img_scale, img=None)
# # stitched_3d[:img_dim[0] // 3]

In [ ]:
# meshes = visuals.create_ellipsoid_meshes(ellip3drecon_axis_a, ellip3drecon_axis_b, ellip3drecon_axis_c)
# visuals.plot_img(img_raw, scale=img_scale, unit=img_unit, meshes=[meshes], mesh_thick=0.5, mesh_alpha=0.1)
# visuals.view_mesh([meshes], img=img_raw, scale=img_scale)

In [ ]:
# visuals.view_3d_vector_field_multiple(
#     vec_pos=[ellip3drecon_axis_a[:, :3], ellip3drecon_axis_b[:, :3], ellip3drecon_axis_c[:, :3]],
#     vec_dir=[ellip3drecon_axis_a[:, 3:], ellip3drecon_axis_b[:, 3:], ellip3drecon_axis_c[:, 3:]],
#     vec_colors=[["red" for _ in range(len(ellip3drecon_axis_a))], ["yellow" for _ in range(len(ellip3drecon_axis_b))],
#                 ["blue" for _ in range(len(ellip3drecon_axis_c))]], vec_length=1, verts=ellip3drecon_axis_a[:, :3],
#     img=img_raw, scale=img_scale,
#     mesh=meshes, mesh_blending="translucent",
#     verts_colors=["white" for _ in range(len(ellip3drecon_axis_a))])

In [ ]:
# visuals.view_3d_vector_field(vec_pos=directors_2dplus[:, :3][:1], vec_dir=directors_2dplus[:, 3:][:1],
#                              vec_colors="red", length=2,
#                              verts=ellip3drecon_axis_a[:, :3], pts_size=3,
#                              verts_colors=["yellow" for _ in range(len(ellip3drecon_axis_a))], img=img_raw,
#                              scale=img_scale)

In [ ]:
# visuals.view_3d_vector_field(vec_pos=directors_2dplus[:, :3], vec_dir=directors_2dplus[:, 3:],
#                              vec_colors=visuals.color_scalar(ar_test, manual_vminmax=[0, crop_max_ar]), length=2,
#                              verts=ellip3drecon_axis_a[:, :3], pts_size=3,
#                              img=img_raw, scale=img_scale,
#                              verts_colors=["yellow" for _ in range(len(ellip3drecon_axis_a))])

In [ ]:
# k = 2 ** 3
# idxs = analysis.coord_search_neighbours(ellip3drecon_axis_a[:, :3], k=k, n_process=10)
# S_reconstructed, n_avg_reconstructed = analysis.avg_3d_nem_tens(directors=ellip3drecon_axis_a, neigh_idxs=idxs)
# visuals.view_3d_vector_field(vec_pos=ellip3drecon_axis_a[:, :3], vec_dir=ellip3drecon_axis_a[:, 3:],
#                              vec_colors=visuals.color_scalar(S_reconstructed, manual_vminmax=[0, 1]),
#                              length=2, verts=ellip3drecon_axis_a[:, :3],
#                              # img=img_raw, scale=img_scale,
#                              verts_colors=visuals.color_scalar(S_reconstructed, manual_vminmax=[0, 1]))
# visuals.plot_hist(S_reconstructed, xlim=[0, 1])

### |1.2| Z-Dependence

In [ ]:
# ==== Plot Z-Dependance of 2D Orientation Analysis Results ====
# fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
# for ax, y, label in zip(axes, [1, 2], ["Mean Angle (°)", "Order Parameter"]):
#     ax.plot(results_2d_slice[:, 0] * img_scale[0], results_2d_slice[:, y], 'o-', label=label)
#     ax.set_ylabel(label)
#     ax.legend()
#     axes[1].set_xlabel(f"Z ({img_unit})")
# plt.savefig(os.path.join(resfig_dir, "2d_slice_nematic_order.pdf"))
# plt.show()

### -- Select Slice --

In [ ]:
z_sel_skel = 0  # img_dim[0] // 2
idx_connect = int(np.argwhere(valid_2dseg_idxs == z_sel_skel)[0][0])
directors_2d_skel = directors_2d_all[idx_connect]
img_selected = img_raw[z_sel_skel].copy()

### |1.3| Find Axis of Elongation

In [ ]:
# ==== Img to be skeletonised ====
z_sel_skel = 0  # img_dim[0] // 2
idx_connect = int(np.argwhere(valid_2dseg_idxs == z_sel_skel)[0][0])
directors_2d_skel = directors_2d_all[idx_connect]
img_selected = img_raw[z_sel_skel].copy()
img_to_skeletonise = analysis.gaussian_blur(img_selected, 60, renorm=True)
img_to_skeletonise = analysis.thresh_img(img_to_skeletonise, thresh=0.4)
img_to_skeletonise = analysis.fill_holes_img(img_to_skeletonise)
img_to_skeletonise = analysis.gaussian_blur(img_to_skeletonise, 0, renorm=True)

# img_to_skeletonise = analysis.fill_holes_img(img_to_skeletonise)
# from scipy.ndimage import binary_dilation
# img_to_skeletonise = binary_dilation(img_to_skeletonise, iterations=120).astype(np.uint8)
# img_to_skeletonise = analysis.gaussian_blur(img_to_skeletonise, 20, renorm=True)
# img_to_skeletonise = analysis.gaussian_blur(img_to_skeletonise, 20, renorm=True)
visuals.plot_matrix(img_to_skeletonise, scale=img_scale, unit=img_unit, cmap='Greys_r', colorbar=True)

In [ ]:
# ==== Find Medial Axis ====
medial_axis_scaled = analysis.find_medial_axis(img=img_to_skeletonise, scale=img_scale)[::100]
s_parallel, s_orthogonal = analysis.proj2curve(directors_2d_skel[:, :2], medial_axis_scaled)
visuals.plot_curve_projections(img=img_selected, scale=img_scale, curve=medial_axis_scaled,
                               pts=directors_2d_skel[:, :2], s_parallel=s_parallel,
                               s_orthogonal=s_orthogonal, unit=img_unit)
datahandler.save_array(medial_axis_scaled, "curve", header="x,y", folderpath=resdata_dir_2dsliced)

In [ ]:
# ==== Fit Spline to Medial Axis ====
spline_order_k = 2
spline_num_pts = 500
spline_smooth = 4.5
spline_sample_interv = 2
medial_start_u, medial_end_u = -2.5, 2.5
medial_curve = analysis.spline_fit_curve(curve=medial_axis_scaled, order_k=spline_order_k, num_pts=spline_num_pts,
                                         smooth=spline_smooth, sample_interv=spline_sample_interv,
                                         start_u=medial_start_u, end_u=medial_end_u)
medial_curve = analysis.filter_curve_inside_shape(medial_curve, img_to_skeletonise, thresh=0.2,
                                                  scale=img_scale[1:])
print(f"Curve points inside shape = {len(medial_curve)}")
s_parallel, s_orthogonal = analysis.proj2curve(directors_2d_skel[:, :2], medial_curve)
visuals.plot_curve_projections(img=img_selected, scale=img_scale, curve=medial_curve, pts=directors_2d_skel[:, :2],
                               s_parallel=s_parallel, s_orthogonal=s_orthogonal,
                               unit=img_unit,
                               savefig=os.path.join(resfig_dir_2dsliced, f"z-{z_sel_skel}_ap-curve_proj.pdf"))
datahandler.save_array(medial_curve, "curve", header="x,y", folderpath=resdata_dir_2dsliced)

In [ ]:
# ==== Use PCA Axis instead ====
pca_axes, pca_center = analysis.find_pca_axes(img_to_skeletonise)
pca_axes[0] = [1, 0]
medial_curve = analysis.draw_pca_curve(pca_center=pca_center, pca_axes=pca_axes, dimensions=img_dim, scale=img_scale)
medial_curve = analysis.filter_curve_inside_shape(medial_curve, img_to_skeletonise, thresh=0.4,
                                                  scale=img_scale[1:])
s_parallel, s_orthogonal = analysis.proj2curve(directors_2d_skel[:, :2], medial_curve)
visuals.plot_curve_projections(img=img_selected, scale=img_scale, curve=medial_curve, pts=directors_2d_skel[:, :2],
                               s_parallel=s_parallel, s_orthogonal=s_orthogonal, unit=img_unit,
                               savefig=os.path.join(resfig_dir_2dsliced, f"z-{z_sel_skel}_ap-curve_proj.pdf"))
datahandler.save_array(medial_curve, "curve", header="x,y", folderpath=resdata_dir_2dsliced)

### -- Load Curve of Elongation --

In [ ]:
# ==== Load PCA Axis instead ====
medial_curve = datahandler.load_array("curve", folderpath=resdata_dir_2dsliced)
# medial_curve = analysis.spline_fit_curve(curve=medial_curve, order_k=2, num_pts=500,
#                                          smooth=4.5, sample_interv=20,
#                                          start_u=0.0, end_u=1.0)
# medial_curve = np.flip(medial_curve, axis=0)
s_parallel, s_orthogonal = analysis.proj2curve(directors_2d_skel[:, :2], medial_curve)
visuals.plot_curve_projections(img=img_selected, scale=img_scale, curve=medial_curve, pts=directors_2d_skel[:, :2],
                               s_parallel=s_parallel, s_orthogonal=s_orthogonal, unit=img_unit,
                               savefig=os.path.join(resfig_dir_2dsliced, f"z-{z_sel_skel}_ap-curve_proj.{file_ending}"))

### |1.4| Accuracy of curve

In [ ]:
from scipy.ndimage import map_coordinates

curve = medial_curve.copy()
# Parameters
profile_half_width = 100  # how far to sample on each side of medial curve
num_samples = 2 * profile_half_width + 1
deltas = np.gradient(curve, axis=0)
tangents = deltas / np.linalg.norm(deltas, axis=1, keepdims=True)
normals = np.stack([-tangents[:, 1], tangents[:, 0]], axis=1)
curve_diffs = np.diff(curve, axis=0)
curve_lengths = np.linalg.norm(curve_diffs, axis=1)
arc_length = np.concatenate([[0], np.cumsum(curve_lengths)])

t = np.linspace(-profile_half_width, profile_half_width, num_samples)
offsets = normals[:, None, :] * t[None, :, None]
sample_points = curve[:, None, :] + offsets
coords = np.stack([sample_points[..., 1], sample_points[..., 0]], axis=0)
coords[0, :, :] /= img_scale[1]
coords[1, :, :] /= img_scale[2]
sampled_profiles = map_coordinates(img_to_skeletonise, coords, order=1, mode='nearest')
profile_sums = sampled_profiles.sum(axis=1, keepdims=True)
profile_sums[profile_sums == 0] = 1
positions = np.linspace(-profile_half_width, profile_half_width, num_samples)
center_of_mass_profiles = (sampled_profiles * positions[None, :]).sum(axis=1) / profile_sums[:, 0]

plt.figure(figsize=(10, 6))
plt.imshow(sampled_profiles.T, aspect='equal',
           extent=[arc_length[0], arc_length[-1], -profile_half_width, profile_half_width],
           origin='lower', cmap="jet")
plt.plot(arc_length, center_of_mass_profiles, color='red', label='Center of Mass')
plt.axhline(0, color='white', linestyle='--', alpha=0.7, label="Midline")
plt.xlabel(f'Distance along curve ({img_unit})')
plt.ylabel(f'Distance orthogonal to curve ({img_unit})')
plt.legend()
plt.savefig(os.path.join(resfig_dir_2dsliced, "curve-accuracy_full.pdf"))
plt.show()

plt.figure()
plt.title(f"Residual ({img_unit})")
plt.plot(center_of_mass_profiles, "o-")
plt.savefig(os.path.join(resfig_dir_2dsliced, "curve-accuracy_residual.pdf"))
plt.show()

visuals.plot_hist(center_of_mass_profiles, title="Residual",
                  savefig=os.path.join(resfig_dir_2dsliced, f"curve-accuracy_residual_hist.{file_ending}"))


### |1.5| Average along binned A-P axes

In [ ]:
num_parallel_bins = 10
num_orthogonal_bins = 5
par_bin_edges = np.linspace(0, analysis.proj2curve(medial_curve, medial_curve)[0].max(), num_parallel_bins + 1)
orth_bin_edges = np.linspace(0, analysis.proj2curve(directors_2d_skel[:, :2], medial_curve)[1].max(),
                             num_orthogonal_bins + 1)
s_parallel_bin_centers = 0.5 * (par_bin_edges[:-1] + par_bin_edges[1:])
s_orthogonal_bin_centers = 0.5 * (orth_bin_edges[:-1] + orth_bin_edges[1:])
dpi_all_figs = 200
ap_par_binned_S_2d_weighted_all = []
ap_orth_binned_S_2d_weighted_all = []
ap_par_binned_S_2d_unweighted_all = []
ap_orth_binned_S_2d_unweighted_all = []
ap_par_binned_n_2d_weighted_all = []
ap_orth_binned_n_2d_weighted_all = []
ap_par_binned_n_2d_unweighted_all = []
ap_orth_binned_n_2d_unweighted_all = []

s_par_orthogonality_weighted_all = []
s_par_orthogonality_unweighted_all = []
s_orth_orthogonality_weighted_all = []
s_orth_orthogonality_unweighted_all = []
zpos_all = []
z_range_binning = np.arange(len(directors_2d_all))
# z_range_binning = [10]
for i in z_range_binning:
    directors_2d_z = directors_2d_all[i]
    zpos = valid_2dseg_idxs[i]
    zpos_all.append(zpos)
    nematic_weights = aspect_ratio_all[i] - 1
    s_parallel, s_orthogonal = analysis.proj2curve(directors_2d_z[:, :2], medial_curve)
    ap_par_binned_idxs = analysis.bin_array_with_indices(s_parallel, num_parallel_bins)
    ap_orth_binned_idxs = analysis.bin_array_with_indices(s_orthogonal, num_orthogonal_bins)

    ap_par_binned_idxs = analysis.bin_indices(s_parallel, par_bin_edges)
    ap_orth_binned_idxs = analysis.bin_indices(s_orthogonal, orth_bin_edges)
    bin_avg_results = analysis.bin_directors(directors=directors_2d_z,
                                             ap_par_binned_idxs=ap_par_binned_idxs,
                                             ap_orth_binned_idxs=ap_orth_binned_idxs,
                                             curve=medial_curve,
                                             nematic_weights=nematic_weights)
    parallel_results, orthogonal_results, nematic_results_weighted, nematic_results_unweighted = bin_avg_results
    ap_par_binned_dirs, s_par_orthogonality_weighted, s_par_orthogonality_unweighted = parallel_results
    ap_orth_binned_dirs, s_orth_orthogonality_weighted, s_orth_orthogonality_unweighted = orthogonal_results
    ap_par_binned_S_2d_weighted, ap_par_binned_n_2d_weighted, ap_orth_binned_S_2d_weighted, ap_orth_binned_n_2d_weighted = nematic_results_weighted
    ap_par_binned_S_2d_unweighted, ap_par_binned_n_2d_unweighted, ap_orth_binned_S_2d_unweighted, ap_orth_binned_n_2d_unweighted = nematic_results_unweighted

    ap_par_binned_S_2d_weighted_all.append(ap_par_binned_S_2d_weighted)
    ap_orth_binned_S_2d_weighted_all.append(ap_orth_binned_S_2d_weighted)
    ap_par_binned_S_2d_unweighted_all.append(ap_par_binned_S_2d_unweighted)
    ap_orth_binned_S_2d_unweighted_all.append(ap_orth_binned_S_2d_unweighted)

    ap_par_binned_n_2d_weighted_all.append(ap_par_binned_n_2d_weighted)
    ap_orth_binned_n_2d_weighted_all.append(ap_orth_binned_n_2d_weighted)
    ap_par_binned_n_2d_unweighted_all.append(ap_par_binned_n_2d_unweighted)
    ap_orth_binned_n_2d_unweighted_all.append(ap_orth_binned_n_2d_unweighted)

    s_par_orthogonality_weighted_all.append(s_par_orthogonality_weighted)
    s_par_orthogonality_unweighted_all.append(s_par_orthogonality_unweighted)
    s_orth_orthogonality_weighted_all.append(s_orth_orthogonality_weighted)
    s_orth_orthogonality_unweighted_all.append(s_orth_orthogonality_unweighted)
    visuals.plot_director_bins(ap_par_binned_dirs=ap_par_binned_dirs, ap_orth_binned_dirs=ap_orth_binned_dirs,
                               ap_par_binned_idxs=ap_par_binned_idxs, ap_orth_binned_idxs=ap_orth_binned_idxs,
                               unit=img_unit, pt_size=50, vmin_vmax_par=[0, num_parallel_bins - 1],
                               vmin_vmax_orth=[0, num_orthogonal_bins - 1],
                               savefig=os.path.join(resfig_dir_2dsliced,
                                                    f"z-{zpos}_bins_parbin-{num_parallel_bins}_orthbin-{num_orthogonal_bins}.{file_ending}"),
                               dpi=dpi_all_figs)

    binned_ap_res_name = f"z-{zpos}_ap-binned_nematic_parbin-{num_parallel_bins}_orthbin-{num_orthogonal_bins}.{file_ending}"
    visuals.plot_binned_ap_results_horizontal(img=img_raw[zpos], curve=medial_curve,
                                              s_parallel_bin_centers=s_parallel_bin_centers,
                                              scale=img_scale, unit=img_unit,
                                              s_orthogonal_bin_centers=s_orthogonal_bin_centers,
                                              ap_par_binned_s_2d_weighted=ap_par_binned_S_2d_weighted,
                                              ap_orth_binned_s_2d_weighted=ap_orth_binned_S_2d_weighted,
                                              s_par_orthogonality_weighted=s_par_orthogonality_weighted,
                                              s_orth_orthogonality_weighted=s_orth_orthogonality_weighted,
                                              savefig=os.path.join(resfig_dir_2dsliced, binned_ap_res_name),
                                              dpi=dpi_all_figs)

### [1.6] Gene Expression vs Elongation

In [ ]:
# img_raw_tbra, img_dim_tbra, img_scale_tbra, img_unit_tbra = analysis.load_img_virtual(path=img_path, c_sel_idx=1)
# tbra_mask = img_raw_tbra[z_sel_skel] > 10
# tbra_pos = np.argwhere(tbra_mask)[:, [1, 0]] * img_scale[1:]
# tbra_vals = img_raw_tbra[z_sel_skel][tbra_mask]
# s_parallel_tbra, s_orthogonal_tbra = analysis.proj2curve(tbra_pos, medial_curve)
# visuals.plot_img(img_raw_tbra, scale=img_scale_tbra, unit=img_unit_tbra,
#                  thresh_mask=analysis.thresh_img(img_raw_tbra, 10))
#
# visuals.plot_curve_projections(img=img_raw_tbra[z_sel_skel], scale=img_scale_tbra, curve=medial_curve, pts=tbra_pos,
#                                s_parallel=s_parallel_tbra, s_orthogonal=s_orthogonal_tbra, unit=img_unit_tbra,
#                                pt_size=0.011,
#                                savefig=os.path.join(resfig_dir_2dsliced, f"TBRA_ap-curve_proj.{file_ending}"))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"{os.path.basename(img_path)} \n Nematic order $S$ | Averaged over {len(zpos_all)} z slices", fontsize=14)

# A-P parallel (left subplot)
sns.boxplot(data=np.array(ap_par_binned_S_2d_weighted_all), ax=ax1, color="blue", boxprops=dict(alpha=.8),
            label="Weighted by $AR-1$")
sns.boxplot(data=np.array(ap_par_binned_S_2d_unweighted_all), ax=ax1, color="red", boxprops=dict(alpha=.5),
            label="Normalised")
ax1.set_xticks(np.arange(len(s_parallel_bin_centers)))
ax1.set_xticklabels(np.round(s_parallel_bin_centers, 2))
ax1.set_xlabel(f"A-P || coordinate ({img_unit})")
ax1.set_ylabel('Nematic order $S$')
ax1.legend(fancybox=True, framealpha=0.5)

# A-P orthogonal (right subplot)
sns.boxplot(data=np.array(ap_orth_binned_S_2d_weighted_all), ax=ax2, color="blue", boxprops=dict(alpha=.8),
            label="Weighted by $AR-1$")
sns.boxplot(data=np.array(ap_orth_binned_S_2d_unweighted_all), ax=ax2, color="red", boxprops=dict(alpha=.5),
            label="Normalised")
ax2.set_xticks(np.arange(len(s_orthogonal_bin_centers)))
ax2.set_xticklabels(np.round(s_orthogonal_bin_centers, 2))
ax2.set_xlabel(f"A-P ⊥ coordinate ({img_unit})")
ax2.set_ylabel('Nematic order $S$')
ax2.legend(fancybox=True, framealpha=0.5)

plt.tight_layout()
plt.show()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"{os.path.basename(img_path)} \n Orthogonality | Averaged over {len(zpos_all)} z slices", fontsize=14)

# A-P parallel (left subplot)
sns.boxplot(data=np.array(s_par_orthogonality_weighted_all), ax=ax1, color="blue", boxprops=dict(alpha=.8),
            label="Weighted by $AR-1$")
sns.boxplot(data=np.array(s_par_orthogonality_unweighted_all), ax=ax1, color="red", boxprops=dict(alpha=.5),
            label="Normalised")
ax1.set_xticks(np.arange(len(s_parallel_bin_centers)))
ax1.set_xticklabels(np.round(s_parallel_bin_centers, 2))
ax1.set_xlabel(f"A-P || coordinate ({img_unit})")
ax1.set_ylabel('Orthogonality')
ax1.legend(fancybox=True, framealpha=0.5)
ax1.set_ylim(-0.1, 1.1)

# A-P orthogonal (right subplot)
sns.boxplot(data=np.array(s_orth_orthogonality_weighted_all), ax=ax2, color="blue", boxprops=dict(alpha=.8),
            label="Weighted by $AR-1$")
sns.boxplot(data=np.array(s_orth_orthogonality_unweighted_all), ax=ax2, color="red", boxprops=dict(alpha=.5),
            label="Normalised")
ax2.set_xticks(np.arange(len(s_orthogonal_bin_centers)))
ax2.set_xticklabels(np.round(s_orthogonal_bin_centers, 2))
ax2.set_xlabel(f"A-P ⊥ coordinate ({img_unit})")
ax2.set_ylabel('Orthogonality')
ax2.legend(fancybox=True, framealpha=0.5)
ax2.set_ylim(-0.1, 1.1)

plt.tight_layout()
plt.show()

## |2| 2D+ Curved Surface

### |2.1| Select and Define surface patches

In [ ]:
# ==== Select Patch Type and Size ====
patch_avg = ["radius", 30]
# patch_avg = ["nearest", 1500]

# ==== Pres-Select Vertices for 2D+ Orientation Analysis ====
idxs_sel = np.arange(layer_mesh.vertices.shape[0])
idxs_sel = analysis.filter_normal_validity(mesh=layer_mesh, idxs_sel=idxs_sel, k=20, threshold=0.99)

# ==== Filter by Intensity Value ====
cutoff_min_intensity = 0.0 * np.max(proj_layer)
cutoff_max_intensity = 1.0 * np.max(proj_layer)
idxs_sel = idxs_sel[(proj_layer[idxs_sel] > cutoff_min_intensity) & (proj_layer[idxs_sel] < cutoff_max_intensity)]

# ==== Compute only points at Interval ====
compute_interval = 100
# idxs_sel = idxs_sel[::compute_interval]
idxs_sel = np.random.choice(idxs_sel, size=int(len(layer_mesh.vertices) / compute_interval))
if len(idxs_sel) == 0:
    print("!! ERROR: No vertices were selected for analysis !!")

# ==== Search Nearest Neighbours ====
patch_type = patch_avg[0]
patch_size = patch_avg[1]

# ==== Tune Plotting parameters ====
if patch_type == "radius":
    idxs_neigh = analysis.coord_search_radius(layer_mesh.vertices, custom_probes=layer_mesh.vertices[idxs_sel],
                                              r=patch_size)
    title_render = f"Extracted directors w.r.t {patch_size}{img_unit}"
elif patch_type == "nearest":
    idxs_neigh = analysis.coord_search_neighbours(layer_mesh.vertices, custom_probes=layer_mesh.vertices[idxs_sel],
                                                  k=patch_size, n_process=8)
    title_render = f"Extracted directors w.r.t {patch_size - 1} neighbours"
else:
    idxs_neigh = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

# # ==== Filter Valid Surface Patches ====
# idxs_neigh, valid_patch_idxs = analysis.filter_valid_patches(verts=layer_mesh.vertices, idxs_neigh=idxs_neigh,
#                                                              factor=0.1)
# idxs_sel = idxs_sel[valid_patch_idxs]

cutoff_intensity_variance = 0.0
# ==== Filter by Intensity Variation ====
proj_layer_variance = np.array([np.var(proj_layer[patch_idxs]) for patch_idxs in idxs_neigh])
visuals.plot_hist(proj_layer_variance, title="Intensity Variance per Patch")

filter_intens_var_mask = proj_layer_variance > cutoff_intensity_variance

# Filter idxs_sel
idxs_sel = idxs_sel[filter_intens_var_mask]

# Filter idxs_neigh (keep only the patches that passed)
idxs_neigh = [patch for i, patch in enumerate(idxs_neigh) if filter_intens_var_mask[i]]

# ==== Find Nearest Intensities ====
proj_layer_neigh = [proj_layer[patch_idxs] for patch_idxs in idxs_neigh]

# ==== Save Vertices for 2D+ Orientation Analysis ====
print(f"Num of directors to be calculated: {len(idxs_sel)} !")
datahandler.save_array(idxs_sel, "calcindeces", header="idx", folderpath=resdata_dir_layer)

# ==== Create the Tangential Bases ====
neighbors_coords = [layer_mesh.vertices[patch] for patch in idxs_neigh]
central_normals = layer_mesh.vertex_normals[idxs_sel]

tan_cords, tan_x, tan_y = analysis.tan_proj(neighbors_coords, central_normals)

# ==== Save the Tangential Bases ====
datahandler.save_array(tan_x, "tan_x", header="t1x,t1y,t1z", folderpath=resdata_dir_layer)
datahandler.save_array(tan_y, "tan_y", header="t2x,t2y,t2z", folderpath=resdata_dir_layer)

In [ ]:
# ==== 3D Render Vertices for 2D+ Orientation Analysis ====
visuals.view_colored_verts(verts=layer_mesh.vertices[idxs_sel],
                           colors=visuals.color_scalar(proj_layer[idxs_sel], cmap="Greens_r"), scale=img_scale)

In [ ]:
# # ==== 3D Render all vertices to be calculated on ====
# all_calculated_mask = np.isin(np.arange(layer_mesh.vertices.shape[0]), idxs_sel)
# all_calculated_color = ["red" if i else "grey" for i in all_calculated_mask]
# visuals.view_colored_mesh(mesh=layer_mesh, vert_colors=all_calculated_color)  #, img=img_raw, scale=img_scale)

### |2.2| Extract Directors

In [ ]:
# ==== Tune Local Orientation Extraction Accuracy ====
grid_N = 30
box_size = 10
debug_2dcurve_analysis = False
debug_vert_idx, debug_grid_x, debug_grid_y, debug_grid_z = None, None, None, None

print(f">> Using local grid of NxN: {grid_N} and box size: {box_size}...")

if debug_2dcurve_analysis:
    # ==== Pick single vertex for debug ====
    debug_vert_idx = np.random.choice(np.arange(idxs_sel.shape[0]))
    big_grid = analysis.tan_interp_batch(
        coords=[tan_cords[debug_vert_idx]],
        intensities=[proj_layer_neigh[debug_vert_idx]],
        grid_size=grid_N
    )[2][0]
    print(big_grid.shape)
else:
    big_grid = np.vstack([grid.T for grid in analysis.tan_interp_batch(
        coords=tan_cords,
        intensities=proj_layer_neigh,
        grid_size=grid_N
    )[2]])

# ==== Extract Directors ====
directors_2dcurved = analysis.batch_2d_orientation(
    big_grid=big_grid, box_size=box_size,
    vertices=layer_mesh.vertices[idxs_sel],
    tan_x=tan_x, tan_y=tan_y,
    debug=debug_2dcurve_analysis,
    debug_idx=debug_vert_idx,
    debug_line_length=1, unit=img_unit
)

if not debug_2dcurve_analysis:
    # ==== Save Directors ====
    datahandler.save_array(directors_2dcurved, "directors_2dcurved", header="x,y,z,vx,vy,vz",
                           folderpath=resdata_dir_layer)

    # ==== Plot Directors ====
    visuals.plot_dir_field(directors=directors_2dcurved, title=title_render,
                           savefig=os.path.join(resfig_dir_layer, "directors_2dcurved.pdf"), veclength=10)

In [ ]:
# ==== 3D Render Directors ====
veclength = 10
vecwidth = 0.5
# visuals.view_3d_vector_field(vec_pos=directors_2dcurved[:, :3], vec_dir=directors_2dcurved[:, 3:], vec_colors="red",
#                              verts=layer_mesh.vertices, verts_colors=visuals.color_scalar(proj_layer, normalise=True, cmap="Greens_r"),
#                              edge_width=veclength / 6, length=veclength, vec_opacity=0.5, pts_size=1, pts_opacity=0.8,
#                              img=None, scale=img_scale)

# visuals.view_3d_vector_field(vec_pos=directors_2dcurved[:, :3], vec_dir=directors_2dcurved[:, 3:], vec_colors="red",
#                              edge_width=veclength / 6, length=veclength)

# visuals.view_mesh_dir_field([layer_mesh], directors=directors_2dcurved, vec_colors="red",
#                             vec_edge_width=veclength / 6, vec_length=veclength)
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved,
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys_r"), vec_length=veclength,
                                    vec_edge_width=vecwidth)

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved[:, :3], directors_2dcurved[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    hexgridsize=400,
    scale_factor=2,
    cmap="Greens",
    arrow_alpha=0.7, figsize=(13, 10),
    savefig=os.path.join(resfig_dir_layer, "spherical_projection_extracted-directors.pdf")
)

### -- Load Directors --

In [ ]:
# ==== Load 2D+ Directors ====
idxs_sel = datahandler.load_array("calcindeces", folderpath=resdata_dir_layer).astype(int)
tan_x = datahandler.load_array("tan_x", folderpath=resdata_dir_layer)
tan_y = datahandler.load_array("tan_y", folderpath=resdata_dir_layer)
directors_2dcurved = datahandler.load_array("directors_2dcurved", folderpath=resdata_dir_layer)

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved[:, :3], directors_2dcurved[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    hexgridsize=400,
    scale_factor=2,
    cmap="Greens",
    arrow_alpha=0.7, figsize=(13, 10)
)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved, vec_edge_width=0.2,
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greens_r"))

### |2.3| Remove Initial Noise by Nematic Averaging

In [ ]:
# ==== Tune Curved Nematic Analysis Number of Neighbours or Radius ====
patch_avg = ["radius", 30]
# patch_avg = ["nearest", 200]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

# ==== Tune Plotting parameters ====
vec_length = 10
vec_edge_width = vec_length / 6
plot2d_view = (20, 0)
renderfigsize = (6, 5)
histfigsize = (4, 3)
veccoords = directors_2dcurved[:, :3]
if patch_type == "radius":
    neigh_idxs = analysis.coord_search_radius(veccoords, r=patch_size)
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    neigh_idxs = analysis.coord_search_neighbours(veccoords, k=patch_size, n_process=8)
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

# ==== Visualise Averaging Patch ====
# patch_sel_idx = np.random.choice(range(len(neigh_idxs)))
# patch_color = np.array(["#FF0000" for _ in range(len(veccoords))])
# patch_color[neigh_idxs[patch_sel_idx]] = "#FFFF00"
# patch_color[neigh_idxs[patch_sel_idx][0]] = "#0000FF"
# visuals.view_colored_verts(verts=veccoords, colors=list(patch_color), use_orig_color=True)

# ==== Calculate Curved Nematic Order ====
S_2dcurv, n_avg_2dcurv = analysis.avg_tan_nem_tens(t1_cov=tan_x, t2_cov=tan_y, directors=directors_2dcurved,
                                                   neigh_idxs=neigh_idxs)

# ==== Save Curved Nematic Order ====
datahandler.save_array(S_2dcurv, name=f"S-order-init_2dcurved_{patch_label}", header="S", folderpath=resdata_dir_layer)
datahandler.save_array(np.column_stack((veccoords, n_avg_2dcurv)), name=f"directors-avg-init_2dcurved_{patch_label}",
                       header="x,y,z,vx,vy,vz", folderpath=resdata_dir_layer)

# ==== Plot Curved Nematic Order ====
directors_2dcurved_avg = directors_2dcurved.copy()
directors_2dcurved_avg[:, 3:] = n_avg_2dcurv
savefig_render = os.path.join(resfig_dir_layer, f"field_intial-avg-nematic_{patch_label}.pdf")
savefig_hist = os.path.join(resfig_dir_layer, f"hist_intial-order-s_{patch_label}.pdf")

visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=vec_length, view_init=plot2d_view, veccolor=S_2dcurv,
                       cmap_label="order scalar $S$", title=title_render, manual_vminmax=[0, 1],
                       savefig=savefig_render, figsize=renderfigsize, show_axes=False)
visuals.plot_hist(array=S_2dcurv, title=title_hist, savefig=savefig_hist,
                  figsize=histfigsize, xlim=[0, 1])
directors_2dcurved_avg_init = directors_2dcurved_avg.copy()

In [ ]:
# ==== 3D Render Curved Nematic Order ====
vec_length = 10
vec_edge_width = 0.5
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys_r"),
                                    vec_length=vec_length, vec_edge_width=vec_edge_width)

# visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
#                                     vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
#                                     mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
#                                                                           cmap="Greys_r"),
#                                     vec_length=vec_length, vec_edge_width=vec_edge_width)
# visuals.view_3d_vector_field(vec_pos=plot_vec_posdir[:, :3], vec_dir=plot_vec_posdir[:, 3:],
#                              vec_colors=visuals.color_scalar(S_2dcurv, cmap="Spectral", manual_vminmax=[0, 1]),
#                              length=vec_length, pts_size=1,
#                              edge_width=vec_edge_width)

# visuals.view_3d_vector_field(vec_pos=plot_vec_posdir[:, :3], vec_dir=plot_vec_posdir[:, 3:],
#                              vec_colors=visuals.color_scalar(S_2dcurv, cmap="Spectral"),
#                              length=vec_length, edge_width=vec_edge_width,
#                              verts=layer_mesh.vertices,
#                              verts_colors=visuals.color_scalar(proj_layer, normalise=True, cmap="Greens_r"))


In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13)
)

### |2.4| Compute nematic order scalar S

In [ ]:
# ==== Tune Curved Nematic Analysis Number of Neighbours ====
patch_avg = ["radius", 30]
# patch_avg = ["nearest", 200]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

# ==== Tune Plotting parameters ====
vec_length = 20
vec_edge_width = vec_length / 6
plot2d_view = (20, 0)
renderfigsize = (6, 5)
histfigsize = (4, 3)

veccoords = directors_2dcurved_avg_init[:, :3]
if patch_type == "radius":
    neigh_idxs = analysis.coord_search_radius(veccoords, r=patch_size)
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    neigh_idxs = analysis.coord_search_neighbours(veccoords, k=patch_size, n_process=8)
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")

# ==== Visualise Averaging Patch ====
# patch_sel_idx = np.random.choice(range(len(neigh_idxs)))
# patch_color = np.array(["#FF0000" for _ in range(len(veccoords))])
# patch_color[neigh_idxs[patch_sel_idx]] = "#FFFF00"
# patch_color[neigh_idxs[patch_sel_idx][0]] = "#0000FF"
# visuals.view_colored_verts(verts=veccoords, colors=list(patch_color), use_orig_color=True)

# ==== Calculate Curved Nematic Order ====
S_2dcurv, n_avg_2dcurv = analysis.avg_tan_nem_tens(t1_cov=tan_x, t2_cov=tan_y, directors=directors_2dcurved_avg_init,
                                                   neigh_idxs=neigh_idxs)

# ==== Save Curved Nematic Order ====
datahandler.save_array(S_2dcurv, name=f"S-order_2dcurved_{patch_label}", header="S", folderpath=resdata_dir_layer)
datahandler.save_array(np.column_stack((veccoords, n_avg_2dcurv)), name=f"directors-avg_2dcurved_{patch_label}",
                       header="x,y,z,vx,vy,vz", folderpath=resdata_dir_layer)

# ==== Plot Curved Nematic Order ====
directors_2dcurved_avg = directors_2dcurved_avg_init.copy()
directors_2dcurved_avg[:, 3:] = n_avg_2dcurv
savefig_render = os.path.join(resfig_dir_layer, f"field_avg-nematic_{patch_label}.pdf")
savefig_hist = os.path.join(resfig_dir_layer, f"hist_order-s_{patch_label}.pdf")

visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=vec_length, view_init=plot2d_view, veccolor=S_2dcurv,
                       cmap_label="order scalar $S$", title=title_render, manual_vminmax=[0, 1],
                       savefig=savefig_render, figsize=renderfigsize, show_axes=False)
visuals.plot_hist(array=S_2dcurv, title=title_hist, savefig=savefig_hist,
                  figsize=histfigsize, xlim=[0, 1])

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=5,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    savefig=os.path.join(resfig_dir_layer, f"spherical_projection_field_avg-nematic_{patch_label}.pdf"),
)

In [ ]:
# ==== 3D Render Curved Nematic Order ====
vec_length = 10
vec_edge_width = 3
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys_r"),
                                    vec_length=vec_length, vec_edge_width=vec_edge_width)
# visuals.view_3d_vector_field(vec_pos=directors_2dcurved_avg[:, :3], vec_dir=directors_2dcurved_avg[:, 3:],
#                              vec_colors=visuals.color_scalar(S_2dcurv, cmap="Spectral", manual_vminmax=[0, 1]),
#                              length=vec_length, pts_size=1,
#                              edge_width=vec_edge_width)
#
# visuals.view_3d_vector_field(vec_pos=directors_2dcurved_avg[:, :3], vec_dir=directors_2dcurved_avg[:, 3:],
#                              vec_colors=visuals.color_scalar(S_2dcurv, cmap="Spectral"),
#                              length=vec_length, edge_width=vec_edge_width,
#                              verts=layer_mesh.vertices,
#                              verts_colors=visuals.color_scalar(proj_layer, normalise=True, cmap="Greens_r"))


### -- Load 2D+ Order --

In [ ]:
# ==== Load 2D+ nematic order ====
patch_avg = ["radius", 30]
# patch_avg = ["nearest", 200]
patch_type = patch_avg[0]
patch_size = patch_avg[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
    title_hist = f"Avg over {patch_size}{img_unit}: Order Scalar $S$"
    title_render = f"Avg over {patch_size}{img_unit}: Average Directors"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
    title_hist = f"Avg over {patch_size - 1} neighbours: Order Scalar $S$"
    title_render = f"Avg over {patch_size - 1} neighbours: Average Directors"
else:
    neigh_idxs = patch_label = title_hist = title_render = None
    print(f"[!] Unknown patch type: {patch_type}")
directors_2dcurved_avg = datahandler.load_array(f"directors-avg_2dcurved_{patch_label}", folderpath=resdata_dir_layer)
S_2dcurv = datahandler.load_array(f"S-order_2dcurved_{patch_label}", folderpath=resdata_dir_layer)
visuals.plot_hist(S_2dcurv, title=title_hist, xlim=[0, 1])
visuals.plot_dir_field(directors=directors_2dcurved_avg, veccolor=S_2dcurv, veclength=4, view_init=(20, 0),
                       cmap_label="order scalar $S$", title=title_render, manual_vminmax=[0, 1], show_axes=False)
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])

visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13)
)

In [ ]:
vec_length = 10
vec_edge_width = 3
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys_r"),
                                    vec_length=vec_length, vec_edge_width=0.2)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greens_r"), vec_length=vec_length,
                                    vec_edge_width=vec_edge_width)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh(layer_mesh,
                          visuals.color_scalar(analysis.interpolate_on_mesh(layer_mesh, idxs_sel, S_2dcurv, k=10),
                                               manual_vminmax=[0, 1]))

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_multiple([layer_mesh, layer_mesh],
                                   [visuals.color_scalar(
                                       analysis.interpolate_on_mesh(layer_mesh, idxs_sel, S_2dcurv, k=30),
                                       manual_vminmax=[0, 1]),
                                       visuals.color_scalar(proj_layer, normalise=True, cmap="Greens_r")])

### |2.5| Identify Defect Location(s) + Geodesic Distance

In [ ]:
# ==== Find Defects and Inter ====
dist_cutoff_defect_localisation = 30
max_candidates_defect_localisation = 15
defect_idxs, rel_dists = analysis.select_geodesic_defects(S_2dcurv, layer_mesh, idxs_sel,
                                                          dist_cutoff=dist_cutoff_defect_localisation, unit=img_unit,
                                                          max_candidates=max_candidates_defect_localisation)
datahandler.save_array(defect_idxs, name="defect-idxs", header="idx", folderpath=resdata_dir_layer)

visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=1, veccolor="red",
                       marker=directors_2dcurved_avg[:, :3][defect_idxs],
                       pt_label="Defect Locations", vec_alpha=0.5, freq=5,
                       pt_color=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"),
                       pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1],
                       savefig=os.path.join(resfig_dir_layer, f"defect-locations.pdf"))

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    marker_idxs=idxs_sel[defect_idxs],
    marker_color=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"),
    savefig=os.path.join(resfig_dir_layer, f"defect-locations.pdf")
)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]), marker_size=500,
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys"),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs]],
                                    marker_colors=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"))

In [ ]:
# ==== Calculate Geodesic Distance between Defects ====
defect_1_index = idxs_sel[defect_idxs[0]]
defect_2_index = idxs_sel[defect_idxs[1]]
dist = analysis.geodesic_distmesh(mesh=layer_mesh, index1=defect_1_index, index2=defect_2_index, debug=True)
visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=0.3,
                       veccolor="red", marker=layer_mesh.vertices[[defect_1_index, defect_2_index]],
                       pt_label="Points", pt_alpha=1.0, view_init=[90, 0])

### -- Load Defect(s) Position(s) --

In [ ]:
defect_idxs = datahandler.load_array(name="defect-idxs", folderpath=resdata_dir_layer).astype(int)
print(f"Found {len(defect_idxs)} defect indeces {defect_idxs}!")
visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=1, veccolor="red",
                       marker=directors_2dcurved_avg[:, :3][defect_idxs],
                       pt_label="Defect Locations", vec_alpha=0.5, freq=5,
                       pt_color=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"),
                       pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1])
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    marker_idxs=idxs_sel[defect_idxs],
    marker_color=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1")
)

In [ ]:
# ==== 3D Render Results ====
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]), marker_size=500,
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys"),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs]],
                                    marker_colors=visuals.color_scalar(np.linspace(0, 1, len(defect_idxs)), "Set1"))

### |2.6| Topological Charge of Nematic Point Defects

In [ ]:
# # ==== Calculate Gaussian Curvature for Topological Charge Analysis ====
gauss_exp = 1 / (np.ptp(layer_mesh.vertices, axis=0).mean() / 2) ** 2
gauss_order_min, gauss_order_max = round(np.log10(gauss_exp)) - 1, round(np.log10(gauss_exp)) + 1
print(f"Gauss should be around {gauss_exp:.2e} (1/{img_unit}^2)")
gauss_crop_range = [0.01 * 10 ** gauss_order_min, 10 ** gauss_order_max]

patch_info = ["radius", 50]
# patch_info = ["nearest", 20]
# gauss_crop_range = None
curv_charge_quick = analysis.curvature_by_srf_fit(mesh=layer_mesh, num_sample=1000, patch_size=patch_info[1],
                                                  patch_mode=patch_info[0], debug=True,
                                                  gauss_crop_range=gauss_crop_range)

gauss_curv_smooth = analysis.interpolate_on_mesh(mesh=layer_mesh, value_idxs=curv_charge_quick[2],
                                                 values=curv_charge_quick[0], k=3)

In [ ]:
visuals.view_colored_mesh(layer_mesh, visuals.color_scalar(gauss_curv_smooth, normalise=True))

In [ ]:
# ==== Calculate Curved Topological Charge ====
patch_charge = ["radius", 20]
# patch_charge = ["nearest", 500]
patch_type = patch_charge[0]
patch_size = patch_charge[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
    charge_patch_idxs = analysis.coord_search_radius(layer_mesh.vertices,
                                                     custom_probes=layer_mesh.vertices[idxs_sel[defect_idxs]],
                                                     r=patch_size)
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
    charge_patch_idxs = analysis.coord_search_neighbours(layer_mesh.vertices,
                                                         custom_probes=layer_mesh.vertices[idxs_sel[defect_idxs]],
                                                         k=patch_size)
else:
    charge_patch_idxs = None
    print(f"[!] Unknown patch type: {patch_type}")

charge_patch_idxs, _ = analysis.filter_valid_patches(layer_mesh.vertices, charge_patch_idxs, factor=0.2)
charge_patch_idxs = analysis.unique_neighborhoods(charge_patch_idxs)
defect_idxs_calc = [np.flatnonzero(idxs_sel == lst[0])[0]
                    for lst in charge_patch_idxs
                    if np.any(idxs_sel == lst[0])]

_, tan_x_all, tan_y_all = analysis.tan_proj(layer_mesh.vertices[:, np.newaxis], layer_mesh.vertex_normals)
m_charge, calc_charge_loop_idxs = analysis.curved_nem_charge(mesh=layer_mesh, directors=directors_2dcurved_avg,
                                                             calc_idxs=defect_idxs_calc,
                                                             director_indeces=idxs_sel,
                                                             tan_x=tan_x_all, tan_y=tan_y_all,
                                                             c_gauss=gauss_curv_smooth,
                                                             loop_angle_precision=1, patch_mode=patch_type,
                                                             patch_size=patch_size, debug=True,
                                                             correct_orientation=True)
print(f"Final number of defects: {len(m_charge)} !")
visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=0.02, veccolor=S_2dcurv,
                       marker=np.vstack([layer_mesh.vertices[i] for i in calc_charge_loop_idxs]),
                       pt_label="Charge Calculation Line",
                       pt_alpha=1.0, cmap_label="order parameter $S$", manual_vminmax=[0, 1])

m_charge_extended = np.full(len(layer_mesh.vertices), 0.0)
if patch_type == "radius":
    charge_patch_idxs_calc = analysis.coord_search_radius(layer_mesh.vertices,
                                                          custom_probes=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
                                                          r=patch_size)
elif patch_type == "nearest":
    charge_patch_idxs_calc = analysis.coord_search_neighbours(layer_mesh.vertices,
                                                              custom_probes=layer_mesh.vertices[
                                                                  idxs_sel[defect_idxs_calc]],
                                                              k=patch_size)
else:
    charge_patch_idxs_calc = None
    print(f"[!] Unknown patch type: {patch_type}")
if isinstance(charge_patch_idxs_calc, list):
    flat_idxs = np.concatenate([np.atleast_1d(np.array(x)) for x in charge_patch_idxs_calc if len(x) > 0])
else:
    flat_idxs = np.ravel(charge_patch_idxs_calc)

m_charge_extended[flat_idxs] = np.repeat(m_charge, [len(np.atleast_1d(x)) for x in charge_patch_idxs_calc])
m_charge_extended = m_charge_extended.ravel()

visuals.plot_dir_field(directors=directors_2dcurved_avg, veclength=10, veccolor=m_charge_extended[idxs_sel],
                       cmap="rainbow",
                       title=r"TOTAL CHARGE$\approx$" + f"{np.nansum(m_charge):.3}",
                       cmap_label="topological charge $m$",
                       manual_vminmax=[-1, 1], show_axes=False, marker=layer_mesh.vertices[idxs_sel[defect_idxs_calc]])

# ==== Save Curved Topological Charge ====
datahandler.save_array(m_charge, name=f"top-charge_2dcurved_{patch_label}", header="m", folderpath=resdata_dir_layer)
datahandler.save_array(defect_idxs_calc, name=f"top-charge_2dcurved_{patch_label}_idxs", header="idx",
                       folderpath=resdata_dir_layer)

# ==== Plot Curved Topological Charge ====
visuals.plot_hist(m_charge, title=f"Sum(m)={np.nansum(m_charge)}",
                  savefig=os.path.join(resfig_dir_layer, f"hist_top-charge_{patch_label}"))



In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="smooth",
                                    mesh_vert_colors="grey",
                                    vec_colors=visuals.color_scalar(m_charge_extended[idxs_sel], manual_vminmax=[-1, 1],
                                                                    cmap="rainbow"),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="smooth",
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="Greys"),
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400)

In [ ]:
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    marker_idxs=idxs_sel[defect_idxs_calc],
    marker_color=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1], cmap="rainbow"),
    savefig=os.path.join(resfig_dir_layer, f"defect-charges.pdf")
)

### -- Load Defect(s) Charge(s) --

In [ ]:
patch_charge = ["radius", 60]
# patch_charge = ["nearest", 200]
patch_type = patch_charge[0]
patch_size = patch_charge[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
else:
    print(f"[!] Unknown patch type: {patch_type}")

m_charge = datahandler.load_array(name=f"top-charge_2dcurved_{patch_label}", folderpath=resdata_dir_layer)
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:])
defect_idxs_calc = datahandler.load_array(name=f"top-charge_2dcurved_{patch_label}_idxs",
                                          folderpath=resdata_dir_layer).astype(int)
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv,
    vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(20, 13),
    marker_idxs=idxs_sel[defect_idxs_calc],
    marker_color=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1], cmap="rainbow"),
    savefig=os.path.join(resfig_dir_layer, f"defect-charges.pdf")
)

### |2.7| Defect Polarisation

In [ ]:
patch_polarisation = ["radius", 20]
# patch_polarisation = ["nearest", 80]
patch_type = patch_polarisation[0]
patch_size = patch_polarisation[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
else:
    print(f"[!] Unknown patch type: {patch_type}")
pol_vecfield, pol_idxs = analysis.compute_defect_polarisations(
    mesh=layer_mesh,
    idxs_sel=idxs_sel,
    directors=directors_2dcurved_avg,
    vertex_normals=layer_mesh.vertex_normals.copy(),
    defect_idxs_calc=defect_idxs_calc,
    m_charge=np.round(m_charge, 2),
    patch_type=patch_type,
    patch_size=patch_size, show_profile=True
)
charge_pol_linked_idxs = np.argsort(defect_idxs_calc)[
    np.searchsorted(defect_idxs_calc, pol_idxs, sorter=np.argsort(defect_idxs_calc))]

datahandler.save_array(pol_vecfield, name=f"def-pol_2dcurved_{patch_label}", header="x,y,z,vx,vy,vz",
                       folderpath=resdata_dir_layer)
datahandler.save_array(charge_pol_linked_idxs, name=f"def-pol_2dcurved_{patch_label}_idxs", header="idx",
                       folderpath=resdata_dir_layer)
# visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="smooth",
#                                     mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
#                                                                           cmap="Greys"),
#                                     vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
#                                     markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
#                                     marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
#                                                                        cmap="rainbow"), marker_size=400,
#                                     marker_vectors=pol_vecfield, marker_vectors_length=10,
#                                     marker_vector_width=10,
#                                     marker_vectors_color=visuals.color_scalar(m_charge[charge_pol_linked_idxs],
#                                                                               manual_vminmax=[-1, 1],
#                                                                               cmap="rainbow"), )

### -- Load Defect(s) Polarisation(s) --

In [ ]:
patch_polarisation = ["radius", 200]
# patch_polarisation = ["nearest", 80]
patch_type = patch_polarisation[0]
patch_size = patch_polarisation[1]

if patch_type == "radius":
    patch_label = f"r-{patch_size}{img_unit}"
elif patch_type == "nearest":
    patch_label = f"k-{patch_size}"
else:
    print(f"[!] Unknown patch type: {patch_type}")

pol_vecfield = datahandler.load_array(name=f"def-pol_2dcurved_{patch_label}", folderpath=resdata_dir_layer)
charge_pol_linked_idxs = datahandler.load_array(name=f"def-pol_2dcurved_{patch_label}_idxs",
                                                folderpath=resdata_dir_layer).astype(int)

In [ ]:
rotation_angles = [1, 1, 1]
sph_proj_phi, sph_proj_theta = analysis.spherical_project(pts=layer_mesh.vertices, rotate=rotation_angles)
vec_dir_phi, vec_dir_theta = analysis.spherical_project_vectors(directors_2dcurved_avg[:, :3],
                                                                directors_2dcurved_avg[:, 3:], rotate=rotation_angles)
pol_dir_phi, pol_dir_theta = analysis.spherical_project_vectors(pol_vecfield[:, :3], pol_vecfield[:, 3:],
                                                                rotate=rotation_angles)
visuals.plot_spherical_projection(
    phi=sph_proj_phi,
    theta=sph_proj_theta,
    intensities=proj_layer,
    vec_pos_phi=sph_proj_phi[idxs_sel],
    vec_pos_theta=sph_proj_theta[idxs_sel],
    vec_dir_phi=vec_dir_phi,
    vec_dir_theta=vec_dir_theta,
    vec_cmap_label="order scalar $S$",
    hexgridsize=400,
    scale_factor=2,
    cmap="Greys_r", vec_width=0.0015,
    veccolor=S_2dcurv, vec_manual_vminmax=[0, 1],
    arrow_alpha=1.0, figsize=(22, 8),
    marker_idxs=idxs_sel[defect_idxs_calc],
    marker_color=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1], cmap="rainbow"),
    marker_vec=(pol_dir_phi, pol_dir_theta, idxs_sel[pol_idxs]),
    marker_vec_scale=20, marker_vec_width=0.005, aspect="equal",
    marker_vec_color=visuals.color_scalar(m_charge[charge_pol_linked_idxs], manual_vminmax=[-1, 1],
                                          cmap="rainbow"),
    savefig=os.path.join(resfig_dir_layer, f"defect-polarisations.pdf")
)

In [ ]:
visuals.view_colored_mesh_dir_field(mesh=layer_mesh, directors=directors_2dcurved_avg, mesh_shading="flat",
                                    mesh_vert_colors=visuals.color_scalar(proj_layer, normalise=True,
                                                                          cmap="inferno"),
                                    vec_colors=visuals.color_scalar(S_2dcurv, manual_vminmax=[0, 1]),
                                    markers=layer_mesh.vertices[idxs_sel[defect_idxs_calc]],
                                    marker_colors=visuals.color_scalar(m_charge, manual_vminmax=[-1, 1],
                                                                       cmap="rainbow"), marker_size=400,
                                    marker_vectors=pol_vecfield, marker_vectors_length=50, vec_length=10,
                                    marker_vector_width=7,
                                    marker_vectors_color=visuals.color_scalar(m_charge[charge_pol_linked_idxs],
                                                                              manual_vminmax=[-1, 1],
                                                                              cmap="rainbow"), )

## |3| 3D Bulk

### |3.1| Directors

In [ ]:
# ==== Extract 3D Directors ====
boxsize_target = 5
boxsize = analysis.rescale_val_xyz(boxsize_target, img_scale)
boxsize = tuple([int(boxsize[i]) for i in range(len(boxsize))])
boxsize = (boxsize_target, boxsize_target, boxsize_target)
print(f"boxsize {boxsize}")
orient3d_result = extra_nematic.compute_3d_orientation(mode="fiber", img=img_raw.astype(np.float64),
                                                       sampling_box_size=boxsize,
                                                       onlydirec=True, upscale_vec=True)
x, y, z = np.meshgrid(
    np.arange(orient3d_result.shape[0]), np.arange(orient3d_result.shape[1]), np.arange(orient3d_result.shape[2]),
    indexing="ij"
)
coordinates = np.stack([x.ravel(), y.ravel(), z.ravel()], axis=1)
veccomponents = orient3d_result.reshape(-1, 3)
raw_directors_3d = np.hstack([coordinates, veccomponents])
print(f"{len(raw_directors_3d)} directors for a {orient3d_result.shape} vector field !")

### |3.2| Threshold Directors

In [ ]:
# ==== Threshold 3D Directors ====
img_thresh_mask = analysis.thresh_img(img=img_raw, thresh=100).copy()
dir_coords = raw_directors_3d[:, :3].astype(int)
directors_3d = raw_directors_3d[img_thresh_mask[tuple(dir_coords.T)] != 0]
num_dirs = 3000
idxs_sel_3d = np.arange(len(directors_3d))
directors_3d = directors_3d[np.random.choice(idxs_sel_3d, size=num_dirs)]
directors_3d[:, :3] *= img_scale
datahandler.save_array(directors_3d, "directors_3d", header="x,y,z,vx,vy,vz", folderpath=resdata_dir)
print(f"Reduced to {len(directors_3d)} directors !")

# ==== Plot 3D Directors ====
# visuals.plot_dir_field(directors=directors_3d,
#                        veclength=1, figsize=(5, 5), view_init=(20, 20),
#                        savefig=os.path.join(resfig_dir, f"directors-3d"))

In [ ]:
# ==== 3D Render 3D Directors ====
visuals.view_3d_vector_field(vec_pos=directors_3d[:, :3], vec_dir=directors_3d[:, 3:],
                             vec_colors="red", length=10, edge_width=0.1, img=img_raw, scale=img_scale)

### -- Load Directors --

In [ ]:
# ==== Load 3D Directors ====
directors_3d = datahandler.load_array(name="directors_3d", folderpath=resdata_dir)

### |3.3| S order

In [ ]:
# ==== Calculate Nematic Order ====
patch_size = 10
idxs = analysis.coord_search_radius(directors_3d[:, :3], r=patch_size)

S_3d, n_avg_3d = extra_nematic.avg_3d_nem_tens(directors=directors_3d, neigh_idxs=idxs)
# directors_3d[:, 3:] = n_avg_3d # [!] overwrites vector field [!]
patch_label = f"r-{patch_size}{img_unit}"
# ==== Save Nematic Order ====
datahandler.save_array(S_3d, name=f"S-order_3d_{patch_label}", header="S", folderpath=resdata_dir)
datahandler.save_array(np.column_stack((directors_3d[:, :3], n_avg_3d)), name=f"directors-avg_3d_{patch_label}",
                       header="x,y,z,vx,vy,vz",
                       folderpath=resdata_dir)

# ==== Plot Nematic Order ====
visuals.plot_dir_field(directors=directors_3d, veccolor=S_3d, cmap_label="order scalar $S$",
                       veclength=10, figsize=(5, 5), view_init=(20, 20),
                       savefig=os.path.join(resfig_dir, f"nematic-field-3d_{patch_label}"))

In [ ]:
# ==== 3D Render Nematic Order ====
vec_length = 5
vec_edge_width = vec_length / 6
freq = 1
plot_coords = directors_3d[:, :3][::freq]
plot_vecs = n_avg_3d[::freq]
plot_colors = visuals.color_scalar(S_3d[::freq], manual_vminmax=[0, 1])
visuals.view_3d_vector_field(vec_pos=plot_coords, vec_dir=plot_vecs,
                             vec_colors=plot_colors, length=vec_length, edge_width=vec_edge_width,
                             img=img_raw, scale=img_scale)

In [ ]:
mask = plot_coords[:, 0] > np.mean(plot_coords, axis=0)[0]
visuals.view_3d_vector_field(vec_pos=plot_coords[mask], vec_dir=plot_vecs[mask],
                             vec_colors=plot_colors[mask], length=vec_length, edge_width=vec_edge_width)